# The Phase-Locked Loop: The Circuit That Built Modern Communications

The phase-locked loop (PLL) is one of the most important building blocks of modern communications and electronics. In a modern smartphone, there are probably in the hundreds of PLLs, some implemented as analog circuits and some through digital signal processing (DSP). In this note, we will explain why PLLs are needed, then what they do, and finally how to implement them in DSP.

## Why are PLLs needed?

### Amplitude Modulation Doesn't Require a PLL

The beginning of wireless communications in 1896 used what we would today call on-off keying (OOK), a special form of amplitude modulation (AM) in which only two amplitudes are used--on (full carrier power) and off (no carrier). [You should recognize this from the ADS-B lab.](https://rfsoc.mit.edu/6S965/F25/assignments/week08) By 1906, technology had progressed to analog AM, allowing for continuously-variable amplitude. Neither of these things required a PLL. Why? Let's examine what an OOK/AM signal looks like:

In [2]:
import numpy as np

sample_frequency_hz = 1e3 # 1 kSps sampling frequency

# Start by defining our message, in this case a 4-symbol-long message ON-OFF-ON-OFF or 0b1010
message = np.array([1, 0, 1, 0], dtype = np.int16)

# Create our baseband signal from the message by making each symbol last a certain amount of time.
symbol_length_samples = 100 # Each symbol lasts 100 samples
baseband = np.repeat(message, symbol_length_samples) # Repeat each symbol symbol_length_samples times

num_samples = len(baseband)

def calculate_am_waveforms(params):
    # Create our carrier signal and modulate it
    carrier_frequency_discrete = params["carrier_frequency_hz"] / sample_frequency_hz
    carrier = np.e ** (1j * 2 * np.pi * carrier_frequency_discrete * np.arange(num_samples))
    modulated_carrier = baseband * carrier

    return {"Baseband": baseband, "Re(Modulated Carrier)": modulated_carrier.real, "Im(Modulated Carrier)": modulated_carrier.imag}

#=============================================IGNORE CODE BELOW HERE===========================================

%matplotlib ipympl
import matplotlib.pyplot as plt
from ipywidgets import AppLayout, FloatSlider, IntSlider, Layout, Box

plt.ioff() # Make Matplotlib only update when explicitly asked to

# Build the user interface
fig, (left_axes, right_axes) = plt.subplots(1, 2)
fig.canvas.header_visible = False
fig.canvas.layout.min_height = '400px'
fig.canvas.layout.min_width = '800px'

carrier_frequency_slider = FloatSlider(
    orientation='horizontal',
    description='Carrier Frequency (Hz):',
    value=10,
    min=10,
    max=100
)

timestep_slider = IntSlider(
    orientation='horizontal',
    description='Timestep:',
    value = 0,
    min = 0,
    max = num_samples - 1
)

vbox_layout = Layout(display = "flex",
                     flex_flow = "column",
                     align_items = "stretch",
                     width = "100%"
                    )
vbox = Box(children = [carrier_frequency_slider, timestep_slider], layout = vbox_layout)

# Function that gets called whenever sliders move
lines = {}
timestep_cursor = None
iq_vector = None
first = True

def update_visualization(change):
    global first
    global timestep_cursor
    global iq_vector
    params = {"carrier_frequency_hz": carrier_frequency_slider.value}
    waveforms = calculate_am_waveforms(params)
    
    if first:
        for waveform_name, waveform in waveforms.items():
            lines[waveform_name] = left_axes.plot(waveform, label = waveform_name)
        # Plot time-domain signals on left axes
        left_axes.legend()
        left_axes.set_xlabel("Timestep (samples)")
        left_axes.set_ylabel("Amplitude (volts)")
        # Plot timestep cursor on left axes (vertical line that shows what time we are viewing)
        timestep_cursor = left_axes.axvline(timestep_slider.value, color = 'r')
        # Plot I/Q in complex plane on right axes
        iq_vector = right_axes.plot([0, 0], [0, 0], "-o", color = 'r')
        right_axes.set_xlabel("Re")
        right_axes.set_ylabel("Im")
        right_axes.set_xlim(-1, 1)
        right_axes.set_ylim(-1, 1)
        right_axes.spines['left'].set_position('center')
        right_axes.spines['bottom'].set_position('center')
        right_axes.spines['right'].set_color('none')
        right_axes.spines['top'].set_color('none')
        right_axes.xaxis.set_ticks_position('bottom')
        right_axes.yaxis.set_ticks_position('left')
        first = False
    else:
        timestep_cursor.set_xdata([timestep_slider.value])
        for waveform_name, waveform in waveforms.items():
            lines[waveform_name][0].set_data(np.arange(len(waveform)), waveform)
        am_waveform_real = waveforms["Re(Modulated Carrier)"]
        am_waveform_imag = waveforms["Im(Modulated Carrier)"]
        iq_vector[0].set_data([0, am_waveform_real[timestep_slider.value]], [0, am_waveform_imag[timestep_slider.value]])

    fig.canvas.draw()
    fig.canvas.flush_events()

# Set up visualization
plt.title("Amplitude Modulation Interactive Demo")

# Update visualization whenever sliders move
carrier_frequency_slider.observe(update_visualization, names = "value")
timestep_slider.observe(update_visualization, names = "value")

# Create layout with plot above sliders
AppLayout(
    center = fig.canvas,
    footer = vbox,
    pane_heights = [0, 6, 1]
)

AppLayout(children=(Box(children=(FloatSlider(value=10.0, description='Carrier Frequency (Hz):', min=10.0), In…

The above interactive visualization has two plots. The left plot shows the time-domain plots of the baseband signal and the modulated AM signal (with a carrier frequency you can vary using the slider below). The right plot shows the I/Q vector at the instant in time set by the timestep slider (indicated by the red cursor on the left plot). The horizontal axis is real/in-phase (I) and the vertical axis is imaginary/quadrature (Q).

Try setting the carrier frequency to a low frequency, say 20 Hz. Move the timestep cursor slowly through one of the pulses of carrier (timesteps 0 to 100). Observe how the I/Q vector has a constant amplitude but spins around at the carrier frequency.

Move the timestep cursor to one of the pulses of no-carrier. Observe how the I/Q vector now has an amplitude of zero.

Experiment with increasing and decreasing the carrier frequency.

Note that changing the carrier frequency does not affect the magnitude of the vector, which is the only thing that actually encodes information in this modulation. We can demodulate just by taking the magnitude:

$$ A(t) = \sqrt{I^2 + Q^2} $$

Why is this important? __It means (in the simplest case with only one signal present) AM does not require us to know anything about the frequency and phase of the carrier.__

### Frequency Modulation (Sorta) Doesn't Require a PLL

Although demodulating AM is very easy (just take the magnitude, which today we do with a CORDIC but people in the past would've done using a diode and a lowpass filter), it has limitations. It is very easily affected by noise and spectrally inefficient (a lot of the transmitter's power is used for the carrier, which carries no information). It also requires linear amplification, as any distortion of the amplitude of the signal in your amplifier will affect the message being transmitted.

By 1933, the state of the art of wireless communication had advanced to frequency modulation (FM). In FM, we keep the amplitude of the signal constant (called "constant-envelope") and vary the frequency. It doesn't make much sense to plot the I/Q vector for FM, since the information is now encoded not by the vector's magnitude but by how fast it spins.

Does FM require use to know anything about the frequency and phase of the carrier? The below visualization lets you explore FM demodulation. It fixes the carrier frequency at 40 Hz, but lets you vary the NCO frequency and phase used to mix down and demodulate the FM.

In [13]:
import numpy as np

def calculate_fm_waveforms(params):
    # Create our carrier signal and modulate it
    carrier_frequency_hz = 40
    carrier_frequency_omega_discrete = (2 * np.pi * carrier_frequency_hz) / sample_frequency_hz # Carrier frequency in discrete
    fm_deviation_hz = 30 # 30 Hz deviation, exaggerated for visual effect
    deviation_factor_kf = 2 * np.pi * fm_deviation_hz # FM deviation in rad/sec
    phase_modulating_signal = (deviation_factor_kf * np.cumsum(baseband)) / sample_frequency_hz # Discrete integral of frequency is phase
    fm_signal_complex = np.e ** (1j * (carrier_frequency_omega_discrete * np.arange(0, len(baseband)) + phase_modulating_signal)) # Complex equivalent of the formula used in the above section

    # Demodulate the FM signal using the NCO
    nco_frequency_omega_discrete = (2 * np.pi * params["nco_frequency_hz"]) / sample_frequency_hz # NCO frequency in discrete
    nco_signal_complex = np.e ** (1j * (nco_frequency_omega_discrete * np.arange(len(baseband)) + params["nco_phase_rad"])) # Generate complex NCO
    fm_mixed_down = fm_signal_complex * nco_signal_complex # Mix FM down to 0 Hz using complex NCO
    fm_demodulated = 0.5 * np.angle(fm_mixed_down[0:-1] * np.conj(fm_mixed_down[1:]))
    
    return {"Baseband": baseband,
            "Re(FM Signal)": fm_signal_complex.real,
            "Re(NCO)": nco_signal_complex.real,
            "Im(NCO)": nco_signal_complex.imag,
            "Demodulated FM": fm_demodulated
    }

#=============================================IGNORE CODE BELOW HERE===========================================

%matplotlib ipympl
import matplotlib.pyplot as plt
from ipywidgets import AppLayout, FloatSlider, IntSlider, Layout, Box

plt.ioff() # Make Matplotlib only update when explicitly asked to

# Build the user interface
fig, axes = plt.subplots(3, 1)
fig.canvas.header_visible = False
fig.canvas.layout.min_height = '400px'
fig.canvas.layout.min_width = '800px'

nco_frequency_slider = FloatSlider(
    orientation='horizontal',
    description='NCO Frequency (Hz):',
    value=40,
    min=30,
    max=50
)

nco_phase_slider = FloatSlider(
    orientation='horizontal',
    description='NCO Phase (rad):',
    value=0,
    min=0,
    max=2 * np.pi
)

vbox_layout = Layout(display = "flex",
                     flex_flow = "column",
                     align_items = "stretch",
                     width = "100%"
                    )
vbox = Box(children = [nco_frequency_slider, nco_phase_slider], layout = vbox_layout)

# Function that gets called whenever sliders move
lines = {}
first = True

def update_visualization(change):
    global first

    params = {"nco_frequency_hz": nco_frequency_slider.value, "nco_phase_rad": nco_phase_slider.value}
    waveforms = calculate_fm_waveforms(params)
    
    if first:
        lines["Re(FM Signal)"] = axes[0].plot(waveforms["Re(FM Signal)"], label = "Re(FM Signal)")
        lines["Baseband"] = axes[0].plot(waveforms["Baseband"], label = "Baseband")
        lines["Re(NCO)"] = axes[1].plot(waveforms["Re(NCO)"], label = "Re(NCO)")
        lines["Im(NCO)"] = axes[1].plot(waveforms["Im(NCO)"], label = "Im(NCO)")
        lines["Demodulated FM"] = axes[2].plot(waveforms["Demodulated FM"], label = "Demodulated FM")

        axes[0].legend()
        axes[1].legend()
        axes[2].legend()
        
        first = False
    else:
        for waveform_name, waveform in waveforms.items():
            lines[waveform_name][0].set_data(np.arange(len(waveform)), waveform)

    fig.canvas.draw()
    fig.canvas.flush_events()

# Set up visualization
axes[0].set_title("Frequency Modulation Interactive Demo")

# Update visualization whenever sliders move
nco_frequency_slider.observe(update_visualization, names = "value")
nco_phase_slider.observe(update_visualization, names = "value")

# Create layout with plot above sliders
AppLayout(
    center = fig.canvas,
    footer = vbox,
    pane_heights = [0, 6, 1]
)

AppLayout(children=(Box(children=(FloatSlider(value=40.0, description='NCO Frequency (Hz):', max=50.0, min=30.…